<a href="https://colab.research.google.com/github/amit-sw/colab_notebooks/blob/main/Structured_Output.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain-openai --quiet
!pip install langchain-google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.5/120.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 3.6 MB/s eta 0:00:00


In [ ]:
import os
from pydantic import BaseModel
from openai import OpenAI

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY']=userdata.get('OPENAI_API_KEY')

AI_MODEL = 'gpt-5-mini'

In [ ]:
system_prompt = """
You are an expert at structured data extraction. You will be given unstructured
text from a news article and should convert it into the given structure.
"""

news_article = """
With an estimated $750 million jackpot, Monday’s Powerball draw will be the 10th
largest in the game’s history after no player matched all six number in
Saturday’s draw, the Multi-State Lottery Association said in a press release.
"""

In [ ]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Using the standard .create() method returns a raw string text response
unstructured_completion = client.chat.completions.create(
    model=AI_MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": news_article}
    ]
)

raw_text = unstructured_completion.choices[0].message.content
print("Raw LLM Response String:")
print(raw_text)
print(f"Data Type of Output: {type(raw_text)}")

Raw LLM Response String:
I don't see a target schema from you — do you want a specific JSON/CSV/other format or different field names? Below I suggest a compact JSON structure and fill it from the sentence. If this is OK I can output a different structure or add/remove fields.

Suggested JSON and extracted values:

{
  "lottery": "Powerball",
  "estimated_jackpot": {
    "amount": 750000000,
    "currency": "USD"
  },
  "draw_day": "Monday",
  "rank_in_history": 10,
  "previous_draw": {
    "day": "Saturday",
    "matched_all_six": false,
    "description": "no player matched all six numbers"
  },
  "source": "Multi-State Lottery Association press release"
}

Tell me if you want different field names, additional fields (e.g., date, region), or another output format.
Data Type of Output: <class 'str'>


In [ ]:
# Define the Pydantic schema
class NewsExtraction(BaseModel):
    lottery: str
    jackpot: int
    organizer: str

In [ ]:
# Swapping out .create() for .parse() and passing response_format
structured_completion = client.chat.completions.parse(
    model=AI_MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": news_article}
    ],
    response_format=NewsExtraction, # Enforces the contract
)

# Extracting the parsed python object directly
native_structured_data = structured_completion.choices[0].message.parsed

print("Parsed Python Object properties:")
print(f"Lottery:   {native_structured_data.lottery}")
print(f"Jackpot:   {native_structured_data.jackpot} (Converted automatically to an Integer!)")
print(f"Organizer: {native_structured_data.organizer}")
print(f"Data Type of Output: {type(native_structured_data)}")

Parsed Python Object properties:
Lottery:   Powerball
Jackpot:   750000000 (Converted automatically to an Integer!)
Organizer: Multi-State Lottery Association
Data Type of Output: <class '__main__.NewsExtraction'>


In [ ]:
# Initializing LangChain's ChatOpenAI wrapper
langchain_client = ChatOpenAI(model=AI_MODEL)

# Formatting messages into LangChain objects
messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content=news_article)
]

# Binding the Pydantic schema to the model using LangChain syntax
langchain_structured_data = langchain_client.with_structured_output(NewsExtraction).invoke(messages)

print("LangChain Parsed Output:")
print(f"Lottery:   {langchain_structured_data.lottery}")
print(f"Jackpot:   {langchain_structured_data.jackpot}")
print(f"Organizer: {langchain_structured_data.organizer}")
print(f"Data Type of Output: {type(langchain_structured_data)}")

LangChain Parsed Output:
Lottery:   Powerball
Jackpot:   750000000
Organizer: Multi-State Lottery Association
Data Type of Output: <class '__main__.NewsExtraction'>


In [ ]:
# Provide your Gemini API Key (Can be generated from Google AI Studio)
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

# Using a standard Gemini production model
AI_MODEL = "gemini-2.5-flash"

In [ ]:
client = ChatGoogleGenerativeAI(api_key=os.getenv("GEMINI_API_KEY"), model=AI_MODEL, temperature=0)

In [ ]:
# LangChain abstracts the backend completely. The syntax here is identical to OpenAI!
news_information = client.with_structured_output(NewsExtraction).invoke(messages)

print("Gemini Structured Output Result:")
print(f"Lottery:   {news_information.lottery}")
print(f"Jackpot:   {news_information.jackpot}")
print(f"Organizer: {news_information.organizer}")
print(f"Data Type: {type(news_information)}")

Gemini Structured Output Result:
Lottery:   Powerball
Jackpot:   750000000
Organizer: Multi-State Lottery Association
Data Type: <class '__main__.NewsExtraction'>
